In [7]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm, colors
import scipy as sp
from sympy import *
%matplotlib ipympl 



In [8]:
delta = 8/9

def A(x):
	return np.sqrt(1-delta**2*(1/np.cosh(x))**2)

r_minus_max = -A(0)
r_plus_min = A(0)

r_minus_max, r_plus_min

(-0.45812284729085123, 0.45812284729085123)

In [9]:
X, Y = np.meshgrid(np.linspace(-2,2,100), np.linspace(-2,2,100))
Z = X + 1j*Y

In [10]:
def plot3D(func, X, Y, Z, title: str):
	plt.close('all')
	fig = plt.figure()
	ax = fig.add_subplot(111, projection='3d')
	# surf = ax.plot_surface(X, Y, np.abs(func(Z)), facecolors=cm.PRGn(np.angle(func(Z))), cmap='PRGn', alpha=0.8, antialiased=True)
	surf = ax.plot_surface(X, Y, np.angle(func(Z)), facecolors=cm.PRGn(np.angle(func(Z))), alpha=0.8, antialiased=True)
	ax.set_xlabel('Re(z)')
	ax.set_ylabel('Im(z)')
	ax.set_zlabel('R²(z)')
	plt.title(title)
	mappable = cm.ScalarMappable(norm=colors.Normalize(vmin=-np.pi, vmax=np.pi), cmap=cm.PRGn)
	fig.colorbar(mappable, ax=ax, shrink=0.5, aspect=10)
	plt.show()

In [11]:
# def R_squared(z):
# 	return (z-r_minus_max)*(z-r_plus_min)

# plot3D(R_squared, X, Y, Z, 'R²(z) = (z - r₋₊)(z - r₊₋) with δ = 8/9',)

# def mysqrt(z):
# 	if np.imag(z) >=0:
# 		return np.sqrt(z)
# 	else:
# 		return -np.sqrt(z)

# vmysqrt=np.vectorize(mysqrt)


# # plot3D(np.sqrt, X, Y, R_squared(Z), 'principal sqrt(R²(z)) with δ = 8/9')
# plot3D(vmysqrt, X, Y, R_squared(Z), 'principal sqrt(R²(z)) with δ = 8/9')

In [13]:
# testing for t=0 at
alpha = r_minus_max
beta = r_plus_min
x0 = 0.0
t0= 0.0
p= 0.0
endpts = [alpha,beta]
endpts


[-0.45812284729085123, 0.45812284729085123]

In [14]:
def integrand(eta,endpts,p, branch:str):
	alpha = endpts[0]
	beta = endpts[1]
	if branch=='neg':
		# return eta**p*np.sqrt(eta-alpha)/np.sqrt(-(1-eta**2)*(eta-beta))*(p+1 + (eta*(3*eta**2-2*beta*eta-1)/(2*(1-eta**2)*(eta-beta))))
		return np.sqrt(eta-alpha)*( ((p+1)*eta**p)/np.sqrt(-(1-eta**2)*(eta - beta)) - eta**(p+1)*(3*eta**2-2*beta*eta-1)/(2*((1-eta**2)*(beta-eta))**(3/2) ))
	
	if branch=='pos':
		# return eta**p*np.sqrt(beta-eta)/np.sqrt((1-eta**2)*(eta-alpha))*(p+1 + (eta*(3*eta**2-2*alpha*eta-1)/(2*(1-eta**2)*(eta-alpha))))
		return np.sqrt(beta - eta)*( ((p+1)*eta**p)/np.sqrt((1-eta**2)*(eta - alpha)) - eta**(p+1)*(-3*eta**2+2*alpha*eta+1)/(2*((1-eta**2)*(eta-alpha))**(3/2)))
	else: 
		return ValueError('Need to specify branch.')


def total_moments(endpts:list,p,x,t, error=False):
	alpha = endpts[0]
	beta = endpts[1]
	
	minus_moment = -2*np.pi*r_minus_max**(p+1)*np.sqrt(r_minus_max - alpha)/np.sqrt(-(1-r_minus_max**2)*(r_minus_max-beta)) + 2*np.pi*sp.integrate.quad(integrand, alpha,r_minus_max, args=(endpts,p,'neg'),points=alpha)[0]
	plus_moment = -2*np.pi*r_plus_min**(p+1)*np.sqrt(beta-r_plus_min)/np.sqrt((1-r_plus_min**2)*(r_plus_min-alpha)) + 2*np.pi*sp.integrate.quad(integrand, r_plus_min,beta, args=(endpts,p,'pos'), points=beta)[0]
	sum_of_moments = minus_moment + plus_moment

	if p==0:
		result = sum_of_moments + 2*np.pi*1j*(x+(alpha+beta)*t)
	
	if p==1:
		result = sum_of_moments + np.pi*1j*((alpha+beta)*x+((3/2)*alpha**2+(3/2)*beta**2+alpha*beta)*t)

	if error==True:
		print(f"The error from numerically integrating the p={p} moment is {sum_of_moments[1]}.")
	
	return result
	# return np.array([np.real(result),np.imag(result).imag])


total_moments(endpts,0,x0,t0), total_moments(endpts,1,x0,t0)


(0j, 0j)

In [ ]:
x_window = 2
t_window = 5
X = np.linspace(-x_window, x_window, 100)
T = np.linspace(0, t_window, 100)

In [16]:

def total_moments_system(endpts,x,t, error=False):
	# alpha = endpts[0]+1j*endpts[1]
	# beta = endpts[2]+1j*endpts[3]
	# realmatrix = np.array([np.real(total_moments([alpha,beta],0,x,t)), np.real(total_moments([alpha,beta],1,x,t))])
	# imagmatrix = np.array([np.imag(total_moments([alpha,beta],0,x,t)), np.imag(total_moments([alpha,beta],1,x,t))])
	# return np.concatenate([realmatrix,imagmatrix])
	matrix = np.array([total_moments(endpts,0,x,t), total_moments(endpts,1,x,t)])
	return matrix

total_moments_system([alpha,beta],x0,t0)


array([0.+0.j, 0.+0.j])

In [ ]:
sp.optimize.newton(total_moments_system, [-A(x0) + 0j,A(x0)+ 0], args=(x0,t0))

/Users/JoanneDong/anaconda3/lib/python3.11/site-packages/scipy/integrate/_quadpack_py.py:589: ComplexWarning: Casting complex values to real discards the imaginary part
  return _quadpack._qagpe(func,a,b,the_points,args,full_output,epsabs,epsrel,limit)


TypeError: Cannot cast array data from dtype('complex128') to dtype('float64') according to the rule 'safe'

array([1.+5.j, 2.+4.j, 3.+3.j])